# 第 1 章 商业问题定义与 ETL 数据接入

本章商业问题：**商城经营数据从哪里来，如何把老板的问题翻译成可分析、可验证、可行动的问题？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [1]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

课程目录: /workspace/projects/practice
ETL API: http://192.168.31.47:38173/api/etl
API 状态: online: eshop-course-etl-api


## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [2]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

可用表数量: 34
addresses 20000 source 用户地址源表：收货地址与省市
admin_action_logs 1 source 后台审计源表：运营动作日志
ads_spend 697 source 广告日花费源表：曝光、点击、转化、消耗
campaigns 56 source 营销活动源表：渠道、类型、目标人群、对照组
cart_items 0 source 购物车行源表
carts 0 source 购物车源表
categories 12 source 商品类目源表：一级类目
coupons 56 source 优惠券模板源表：门槛、面额、发放与使用计数
inventory_movements 258073 source 库存流水源表：初始库存、补货、销售出库
order_items 250682 source 订单行源表：SKU、数量、单价、成本、分摊优惠
orders 117114 source 订单头源表：渠道、活动、金额、状态
page_events 1036466 source 行为事件源表：曝光/浏览/搜索/加购/结算/支付成功


## 2. 指标口径不是细节，而是管理共识

同样叫 GMV，不同公司可能有不同口径。本课程要求每个指标都能说清楚定义。

In [3]:
metrics = get_metrics()["metrics"]
for key in ["gmv", "orderCount", "buyerCount", "avgOrderValue"]:
    print(key, metrics[key]["value"], "| 口径:", metrics[key]["definition"])

funnel = metrics["funnel"]
stages = ["view_home", "view_product", "add_to_cart", "checkout", "pay_success"]
print("\n流量漏斗:")
for s in stages:
    print(s, funnel.get(s))

gmv 156027298.84 | 口径: SUM(orders.paid_amount) WHERE status IN ('paid', 'completed')
orderCount 108043 | 口径: COUNT(DISTINCT order_id) WHERE status IN ('paid', 'completed')
buyerCount 18118 | 口径: COUNT(DISTINCT user_id) WHERE status IN ('paid', 'completed')
avgOrderValue 1444.12 | 口径: GMV / 订单数

流量漏斗:
view_home 237111
view_product 191556
add_to_cart 150931
checkout 126787
pay_success 117111


## 3. 数据质量检查

数据质量检查对应商业上的可信度。如果订单主键重复、金额异常、日期缺失，后面的模型再漂亮也不值得信。

In [4]:
quality = get_quality_report()
print("检查总数:", quality["summary"]["total"])
print("通过:", quality["summary"]["pass"], "警告:", quality["summary"]["warn"], "失败:", quality["summary"]["fail"])
for item in quality["checks"][:5]:
    print(item["category"], item["name"], item["status"], item["detail"])

检查总数: 18
通过: 17 警告: 1 失败: 0
完整性 users 行数 pass 实际 20,000 行，预期约 20,000 行
完整性 orders 行数 pass 实际 117,114 行，预期约 100,000 行
完整性 page_events 行数 pass 实际 1,036,466 行，预期约 700,000 行
完整性 sku 行数 pass 实际 864 行，预期约 864 行
完整性 campaigns 行数 pass 实际 56 行，预期约 50 行


In [5]:
schema = get_schema("fact_order")
print("fact_order 字段:")
for col in schema["columns"]:
    print(col["name"], col["type"])
assert any(c["name"] == "paid_amount" for c in schema["columns"])
print("第 01 章验证通过")

fact_order 字段:
order_id TEXT
user_id TEXT
campaign_id TEXT
order_date 
channel TEXT
status TEXT
subtotal REAL
discount_amount REAL
shipping_fee REAL
total_amount REAL
paid_amount REAL
第 01 章验证通过


## 学生作业

请补充 150 到 300 字商业解读，至少引用两个数据结果，并给出一个明确运营动作。